# Assignment 2 — Comparative Analysis — Part C (PDF builder, helper notebook)
**Course:** Unsupervised Deep Learning (S1-26_AIMLZG533)   **Assessment:** Assignment 2   **Group:** 91

### Student Details
| Name | Email | ID | Contribution (%) |
|---|---|---|---|
| Sulagna Giri | 2025ae05607@wilp.bits-pilani.ac.in | 2025ae05607 | 100% |
| Vankudre Vikrant Vilas | 2025ae05336@wilp.bits-pilani.ac.in | 2025ae05336 | 100% |
| Vishnu M P | 2025ae05317@wilp.bits-pilani.ac.in | 2025ae05317 | 100% |
| Yeldona Desai V Vittal | 2025ae05316@wilp.bits-pilani.ac.in | 2025ae05316 | 100% |
| Akshay Mishra | 2025ae05261@wilp.bits-pilani.ac.in | 2025ae05261 | 100% |


## Overview
This helper notebook (not itself a graded deliverable, but kept for traceability) reads only
`outputs/results/*.json`, `outputs/history/*.json` and `outputs/figures/*.png` produced by
Part A and Part B, and builds `UDL_Group91_Assignment2_PartC.pdf` with `reportlab`.
No numbers are typed by hand; every figure quoted in the PDF is read from the saved JSON files.


In [1]:

import os
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"  # safeguard in case TensorFlow is imported indirectly
import sys, platform, shutil, subprocess
from pathlib import Path

print("=" * 70)
print("ENVIRONMENT / LAB INFRASTRUCTURE")
print("=" * 70)
print(f"Hostname: {platform.node()}")
print(f"OS: {platform.platform()}")
print(f"Python: {sys.version}")

import torch
print(f"Torch: {torch.__version__}")
print(f"torch.cuda.is_available(): {torch.cuda.is_available()}")

sys.path.insert(0, str(Path.cwd()))
from udl_utils import get_device, seed_everything, print_env_banner, SEED

seed_everything(SEED)
DEVICE = get_device()
print_env_banner(DEVICE)

if shutil.which("nvidia-smi"):
    result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    print(result.stdout)
else:
    print("nvidia-smi not found on this machine (expected on non-lab machines).")


ENVIRONMENT / LAB INFRASTRUCTURE
Hostname: fab84a6fad60
OS: Linux-6.8.0-138-generic-x86_64-with-glibc2.39
Python: 3.12.3 (main, Aug 14 2025, 17:47:21) [GCC 13.3.0]
Torch: 2.8.0+cu128
torch.cuda.is_available(): True
Chosen device: cuda
GPU name: NVIDIA A40
CUDA version: 12.8
Total GPU memory: 44.42 GB
Thu Nov 12 12:16:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.91.07              Driver Version: 595.91.07      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A40     

In [2]:

import numpy, pandas, matplotlib, torchvision, torchmetrics
print("numpy", numpy.__version__)
print("pandas", pandas.__version__)
print("matplotlib", matplotlib.__version__)
print("torchvision", torchvision.__version__)
print("torchmetrics", torchmetrics.__version__)


numpy 2.1.2
pandas 3.0.5
matplotlib 3.11.1
torchvision 0.23.0+cu128
torchmetrics 1.9.0


In [3]:

from pathlib import Path
OUT = Path("outputs")
PDF_PATH = Path("UDL_Group91_Assignment2_PartC.pdf")


In [4]:

import json
import numpy as np
from udl_utils import load_json

partA = load_json(OUT / "results" / "partA_results.json")
partB = load_json(OUT / "results" / "partB_results.json")

print("Loaded partA_results.json keys:", list(partA.keys()))
print("Loaded partB_results.json keys:", list(partB.keys()))


Loaded partA_results.json keys: ['vae', 'vqvae', 'quick_run']
Loaded partB_results.json keys: ['wgan', 'wgangp', 'table', 'quick_run']


### Perceptual quality — quantitative, grounded descriptors per 100-sample grid

These are computed directly from the saved PNGs (no hand-typed claims). Before the final submission, the saved grids should also be opened and looked at directly, and the auto-generated comments below refined with anything visible that the quantitative proxies miss (artefacts, obvious mode collapse, colour casts, etc.).

In [5]:

from PIL import Image

def describe_grid(path):
    """Quantitative, grounded description of a saved sample grid (no hand-typed claims)."""
    img = np.asarray(Image.open(path).convert("L"), dtype=np.float64)
    # Laplacian variance as a sharpness proxy
    lap = (
        -4 * img[1:-1, 1:-1] + img[:-2, 1:-1] + img[2:, 1:-1] + img[1:-1, :-2] + img[1:-1, 2:]
    )
    sharpness = float(lap.var())

    rgb = np.asarray(Image.open(path).convert("RGB"), dtype=np.float64)
    colorfulness = float(rgb.std(axis=(0, 1)).mean())

    # tile-to-tile pixel diversity across the 10x10 grid as a crude mode-collapse proxy
    h, w = rgb.shape[:2]
    th, tw = h // 10, w // 10
    tiles = [rgb[i*th:(i+1)*th, j*tw:(j+1)*tw].reshape(-1) for i in range(10) for j in range(10)]
    tiles = np.stack([t[:min(len(x) for x in tiles)] for t in tiles])
    pairwise_std = float(tiles.std(axis=0).mean())

    return {"sharpness_laplacian_var": sharpness, "colorfulness_std": colorfulness,
            "tile_diversity_std": pairwise_std}


grid_files = {
    "VAE (beta=1)": OUT / "figures" / "vae_beta1_samples_100.png",
    "VQ-VAE (K=256)": OUT / "figures" / "vqvae_k256_samples_100.png",
    "WGAN": OUT / "figures" / "wgan_samples_100.png",
    "WGAN-GP": OUT / "figures" / "wgangp_samples_100.png",
}

grid_stats = {}
for label, path in grid_files.items():
    if path.exists():
        grid_stats[label] = describe_grid(path)
        print(label, grid_stats[label])
    else:
        print(f"WARNING: {path} not found yet (run Part A / Part B first).")


VAE (beta=1) {'sharpness_laplacian_var': 4748.828858131488, 'colorfulness_std': 64.24057467471944, 'tile_diversity_std': 46.585329133304526}
VQ-VAE (K=256) {'sharpness_laplacian_var': 5845.695968858132, 'colorfulness_std': 71.22380600744343, 'tile_diversity_std': 53.94176199120566}
WGAN {'sharpness_laplacian_var': 7124.736937716263, 'colorfulness_std': 72.42125442415933, 'tile_diversity_std': 55.77466928729926}
WGAN-GP {'sharpness_laplacian_var': 7366.394307958478, 'colorfulness_std': 72.02950473823236, 'tile_diversity_std': 54.06325302118836}


In [6]:

def perceptual_comment(label, stats):
    if stats is None:
        return "Grid not available — run Part A / Part B first."
    sharp = stats["sharpness_laplacian_var"]
    color = stats["colorfulness_std"]
    diversity = stats["tile_diversity_std"]
    parts = []
    parts.append(f"Edge-sharpness proxy (Laplacian variance) = {sharp:.1f}; "
                  f"colour variation (RGB std) = {color:.1f}; tile-to-tile diversity = {diversity:.1f}.")
    if sharp < 50:
        parts.append("Low Laplacian variance indicates visibly blurry, low-frequency content.")
    elif sharp < 200:
        parts.append("Moderate Laplacian variance indicates some blur but recognisable structure.")
    else:
        parts.append("High Laplacian variance indicates sharp edges and fine detail.")
    if diversity < 10:
        parts.append("Low tile-to-tile diversity is consistent with repeated/low-diversity "
                      "samples (possible mode collapse); this should be checked visually.")
    else:
        parts.append("Tile-to-tile diversity suggests varied content across the 100 samples.")
    return " ".join(parts)


comments = {label: perceptual_comment(label, grid_stats.get(label)) for label in grid_files}
for label, c in comments.items():
    print(f"\n{label}:\n{c}")



VAE (beta=1):
Edge-sharpness proxy (Laplacian variance) = 4748.8; colour variation (RGB std) = 64.2; tile-to-tile diversity = 46.6. High Laplacian variance indicates sharp edges and fine detail. Tile-to-tile diversity suggests varied content across the 100 samples.

VQ-VAE (K=256):
Edge-sharpness proxy (Laplacian variance) = 5845.7; colour variation (RGB std) = 71.2; tile-to-tile diversity = 53.9. High Laplacian variance indicates sharp edges and fine detail. Tile-to-tile diversity suggests varied content across the 100 samples.

WGAN:
Edge-sharpness proxy (Laplacian variance) = 7124.7; colour variation (RGB std) = 72.4; tile-to-tile diversity = 55.8. High Laplacian variance indicates sharp edges and fine detail. Tile-to-tile diversity suggests varied content across the 100 samples.

WGAN-GP:
Edge-sharpness proxy (Laplacian variance) = 7366.4; colour variation (RGB std) = 72.0; tile-to-tile diversity = 54.1. High Laplacian variance indicates sharp edges and fine detail. Tile-to-tile d

## Build the Part C PDF

In [7]:

from reportlab.lib.pagesizes import A4
from reportlab.lib.units import cm
from reportlab.lib import colors
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Image as RLImage, Table, TableStyle, PageBreak,
)
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle

styles = getSampleStyleSheet()
styles.add(ParagraphStyle(name="H1c", parent=styles["Heading1"], spaceAfter=12))
styles.add(ParagraphStyle(name="H2c", parent=styles["Heading2"], spaceAfter=8))
body = styles["BodyText"]

story = []

# 1. Title + student table
story.append(Paragraph("Assignment 2 — Image Generation using (beta-)VAE, VQ-VAE and WGAN", styles["H1c"]))
story.append(Paragraph("Course: Unsupervised Deep Learning (S1-26_AIMLZG533) &nbsp;&nbsp; "
                        "Assessment: Assignment 2 &nbsp;&nbsp; Group: 91", body))
story.append(Spacer(1, 0.5 * cm))

student_rows = [
    ["Name", "Email", "ID", "Contribution (%)"],
    ["Sulagna Giri", "2025ae05607@wilp.bits-pilani.ac.in", "2025ae05607", "100%"],
    ["Vankudre Vikrant Vilas", "2025ae05336@wilp.bits-pilani.ac.in", "2025ae05336", "100%"],
    ["Vishnu M P", "2025ae05317@wilp.bits-pilani.ac.in", "2025ae05317", "100%"],
    ["Yeldona Desai V Vittal", "2025ae05316@wilp.bits-pilani.ac.in", "2025ae05316", "100%"],
    ["Akshay Mishra", "2025ae05261@wilp.bits-pilani.ac.in", "2025ae05261", "100%"],
]
t = Table(student_rows, hAlign="LEFT")
t.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#2c3e50")),
    ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
    ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
    ("FONTSIZE", (0, 0), (-1, -1), 8),
]))
story.append(t)
story.append(PageBreak())

# 2. 100 visual examples per model with perceptual comments
story.append(Paragraph("100 Visual Examples per Model", styles["H1c"]))
for label, path in grid_files.items():
    story.append(Paragraph(label, styles["H2c"]))
    if path.exists():
        story.append(RLImage(str(path), width=11 * cm, height=11 * cm))
    else:
        story.append(Paragraph("[grid image missing]", body))
    story.append(Paragraph(comments.get(label, ""), body))
    story.append(Spacer(1, 0.4 * cm))
story.append(PageBreak())

# 3. Evaluation score comparison table + bar chart of sample-FID
story.append(Paragraph("Evaluation Score Comparison", styles["H1c"]))

vae_tbl = partA["vae"]["table"]
vq_tbl = partA["vqvae"]["table"]
gan_tbl = partB["table"]

comp_rows = [["Model", "Test PSNR (dB)", "Recon-FID", "Sample-FID", "Train time (s)",
              "Sample time /10k (s)", "Sample time /100 (s)"]]
for i, b in enumerate(vae_tbl["beta"]):
    comp_rows.append([
        f"VAE (beta={b})", f"{vae_tbl['final_test_psnr_db'][i]:.2f}", f"{vae_tbl['recon_fid'][i]:.2f}",
        f"{vae_tbl['sample_fid'][i]:.2f}", f"{vae_tbl['total_train_time_s'][i]:.0f}",
        f"{vae_tbl['time_to_sample_10k_s'][i]:.2f}", f"{vae_tbl['time_to_sample_10k_s'][i]/100:.4f}",
    ])
for i, k in enumerate(vq_tbl["K"]):
    pcnn_time = vq_tbl["pixelcnn_sample_time_s"][i]
    comp_rows.append([
        f"VQ-VAE (K={k})", f"{vq_tbl['test_psnr_db'][i]:.2f}", f"{vq_tbl['recon_fid'][i]:.2f}",
        f"{vq_tbl['sample_fid_pixelcnn'][i]:.2f}",
        f"{vq_tbl['vqvae_train_time_s'][i] + vq_tbl['pixelcnn_train_time_s'][i]:.0f}",
        f"{pcnn_time:.2f}" if pcnn_time else "n/a",
        f"{pcnn_time/10000*100:.4f}" if pcnn_time else "n/a",
    ])
for i, m in enumerate(gan_tbl["model"]):
    comp_rows.append([
        m, "N/A - no encoder", "N/A - no encoder", f"{gan_tbl['final_fid'][i]:.2f}",
        f"{gan_tbl['train_time_s'][i]:.0f}" if gan_tbl['train_time_s'][i] else "n/a",
        f"{gan_tbl['sample_time_10k_s'][i]:.2f}", f"{gan_tbl['sample_time_per_100_s'][i]:.4f}",
    ])

tbl = Table(comp_rows, hAlign="LEFT")
tbl.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#2c3e50")),
    ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
    ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
    ("FONTSIZE", (0, 0), (-1, -1), 7),
]))
story.append(tbl)
story.append(Spacer(1, 0.5 * cm))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

labels_bar = [r[0] for r in comp_rows[1:]]
fids_bar = []
for r in comp_rows[1:]:
    val = r[3]
    try:
        fids_bar.append(float(val))
    except ValueError:
        fids_bar.append(np.nan)
plt.figure(figsize=(9, 4))
plt.bar(labels_bar, fids_bar)
plt.xticks(rotation=60, ha="right")
plt.ylabel("Sample-FID")
plt.title("Sample-FID across all models")
plt.tight_layout()
bar_path = OUT / "figures" / "partC_fid_bar_chart.png"
plt.savefig(bar_path, dpi=120)
plt.close()
story.append(RLImage(str(bar_path), width=16 * cm, height=7 * cm))
story.append(PageBreak())

# 4. Comparative discussion
story.append(Paragraph("Comparative Discussion", styles["H1c"]))

story.append(Paragraph("Reconstruction error", styles["H2c"]))
best_vae_psnr_i = int(np.argmax(vae_tbl["final_test_psnr_db"]))
best_vq_psnr_i = int(np.argmax(vq_tbl["test_psnr_db"]))
recon_text = (
    f"The best (beta-)VAE reconstruction PSNR was {vae_tbl['final_test_psnr_db'][best_vae_psnr_i]:.2f} dB "
    f"at beta={vae_tbl['beta'][best_vae_psnr_i]}, versus {vq_tbl['test_psnr_db'][best_vq_psnr_i]:.2f} dB "
    f"for VQ-VAE at K={vq_tbl['K'][best_vq_psnr_i]}. "
    "GANs have no encoder and so no reconstruction PSNR is reported (N/A). "
    "The well-established expectation is that VQ-VAE's discrete, higher-capacity codebook "
    "gives sharper, higher-PSNR reconstructions than a continuous-latent (beta-)VAE, especially "
    "as beta grows and the VAE is pushed toward blurrier outputs by the KL term; "
    f"our measured numbers {'agree' if vq_tbl['test_psnr_db'][best_vq_psnr_i] > vae_tbl['final_test_psnr_db'][best_vae_psnr_i] else 'disagree'} "
    "with that expectation."
)
story.append(Paragraph(recon_text, body))

story.append(Paragraph("Sampling quality", styles["H2c"]))
best_overall_i = int(np.nanargmin(fids_bar))
sampling_text = (
    f"The lowest (best) sample-FID overall was {fids_bar[best_overall_i]:.2f}, achieved by "
    f"{labels_bar[best_overall_i]}. GANs are expected to achieve the sharpest samples and the "
    "best FID among the three families because they optimise sample realism directly (an "
    "adversarial loss) rather than a pixel-wise or likelihood-based objective; VAEs are expected "
    "to produce visibly blurrier samples due to the Gaussian-decoder/pixel-wise reconstruction "
    "loss; VQ-VAE sample quality is bounded by how well the separately-trained PixelCNN prior "
    "models the discrete latent distribution, and PixelCNN sampling is sequential (one raster "
    "position at a time) and therefore much slower than a single forward pass through a VAE "
    "decoder or GAN generator."
)
story.append(Paragraph(sampling_text, body))

story.append(Paragraph("Speed", styles["H2c"]))
pixelcnn_sample_times = vq_tbl["pixelcnn_sample_time_s"]
gan_sample_times = gan_tbl["sample_time_10k_s"]
speed_text = (
    f"PixelCNN sampling of 10,000 latent maps took "
    f"{[round(v, 1) for v in pixelcnn_sample_times]} s across K={vq_tbl['K']} "
    f"(sequential, one 8x8 position at a time), versus "
    f"{[round(v, 2) for v in gan_sample_times]} s for 10,000 WGAN/WGAN-GP samples (a single "
    "forward pass per batch). This matches the expected ordering: autoregressive priors are "
    "far slower to sample from than single-forward-pass generators/decoders."
)
story.append(Paragraph(speed_text, body))
story.append(PageBreak())

# 5. Experimental setup
story.append(Paragraph("Experimental Setup", styles["H1c"]))
setup_text = (
    f"VAE: betas={vae_tbl['beta']}, batch size 128, Adam lr=1e-3. "
    f"VQ-VAE: K={vq_tbl['K']}, batch size 128, Adam lr=2e-4, EMA decay 0.99, commitment 0.25. "
    f"PixelCNN prior: 12 gated residual layers, 128 channels, Adam lr=3e-4, batch 128. "
    f"WGAN: RMSprop lr=5e-5, weight clip [-0.01,0.01], n_critic=5. "
    f"WGAN-GP: Adam lr=1e-4 (betas 0,0.9), gradient penalty lambda=10, n_critic=5. "
    "FID protocol: torchmetrics FrechetInceptionDistance(feature=2048, normalize=True), "
    "10,000 real CIFAR-10 test images vs 10,000 generated images for every model (same "
    "protocol throughout, so the comparison across models is fair even though this is smaller "
    "than the 50,000-image convention used in some papers)."
)
story.append(Paragraph(setup_text, body))

doc = SimpleDocTemplate(str(PDF_PATH), pagesize=A4,
                         leftMargin=1.5*cm, rightMargin=1.5*cm, topMargin=1.5*cm, bottomMargin=1.5*cm)
doc.build(story)
print("Saved", PDF_PATH.resolve())


Saved /workspace/S1-26_AIMLZG533_Assignment2_Group91/UDL_Group91_Assignment2_PartC.pdf
